# 🗺️ Análise Estrutural da Malha Viária de Natal/RN
### Planejamento de Corredores de Metrô com Teoria dos Grafos
#### Discente: Kawan Vinicius Feitosa de Oliveira
---
**Disciplina:** DCA3702 - ALGORITMOS E ESTRUTURAS DE DADOS II

**Objetivo:** Identificar os elementos estruturais mais importantes da malha viária de Natal/RN usando métricas de grafos — grau, betweenness, closeness e k-core — para subsidiar uma proposta de traçado de metrô.

---

## 📐 Estrutura deste Notebook

| Seção | Conteúdo |
|-------|----------|
| 0 | Instalação de dependências |
| 1 | Coleta da rede viária (OSMnx) |
| 2 | Métricas estruturais (NetworkX) |
| 3 | Análise e 7 perguntas obrigatórias |
| 4 | Visualizações (matplotlib + folium) |
| 5 | Exportação para Gephi (.graphml) |

---

## 💡 Decisão de projeto: por que Natal/RN?

Natal, curiosamente, é a **única capital brasileira sem metrô** em operação plena. Além de tudo, a cidade em si possui uma configuração espacial desafiadora: o **Rio Potengi** divide a malha urbana entre Zona Norte e Zona Sul, gerando gargalos de mobilidade bem identificáveis via métricas de betweenness; gerando, praticamente, dois grandes polos. Isso torna Natal um caso de estudo ideal para demonstrar como **teoria dos grafos pode orientar decisões de infraestrutura urbana, e melhorar a mobilidade em grandes cidades**.

## Seção 0 — Instalação de Dependências

Execute esta célula apenas na primeira vez ou após reiniciar o ambiente aqui. Esta célula irá baixar todos os pacotes necessários pra o notebook funcionar.

In [22]:
# ─── Instalação (Google Colab) ───────────────────────────────────────────────
import sys

PACOTES = [
    "osmnx",           # download e manipulação de redes viárias do OpenStreetMap
    "networkx",        # cálculo de métricas de grafos
    "folium",          # mapas interativos HTML
    "matplotlib",      # visualizações estáticas
    "seaborn",         # estilo de gráficos
    "geopandas",       # operações geoespaciais (dependência do osmnx)
]

import subprocess
for p in PACOTES:
    subprocess.run([sys.executable, "-m", "pip", "install", p, "-q"], check=True)

print("✓ Todas as dependências instaladas com sucesso.")

✓ Todas as dependências instaladas com sucesso.


## Seção 1 — Coleta da Rede Viária (OSMnx)

### O que é o OSMnx?
O **OSMnx** é uma biblioteca Python que baixa dados do **OpenStreetMap (OSM)** e os converte diretamente em grafos NetworkX. Ele cuida de todo o processo de geocodificação, download de geometrias e simplificação da rede.

### Por que `network_type="drive"`?
Utilizamos o tipo `"drive"` porque:
- O metrô compartilhará corredores com a infraestrutura viária principal;
- Ciclovias e calçadas não são relevantes para o traçado de metrô;
- A rede `drive` inclui avenidas, ruas e vias de acesso — exatamente o que precisamos para analisar mobilidade urbana motorizada.

### Estrutura do grafo resultante
- **Nós**: interseções viárias (cruzamentos, rotatórias, entroncamentos)
- **Arestas**: segmentos de via entre duas interseções
- **Atributos dos nós**: `x` (longitude), `y` (latitude) — usados no Gephi
- **Atributos das arestas**: `length` (metros), `name`, `oneway`, `speed_kph`

In [25]:
import os
import osmnx as ox
import networkx as nx
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

# ─── Parâmetros ───────────────────────────────────────────────────────────────
CIDADE       = "Natal, Rio Grande do Norte, Brazil"
NETWORK_TYPE = "drive"
CACHE_FILE   = "natal_drive_bruto.graphml"

# Cria pastas de saída
os.makedirs("graficos", exist_ok=True)

# ─── Download (com cache) ─────────────────────────────────────────────────────
if os.path.exists(CACHE_FILE):
    print(f"Carregando grafo do cache '{CACHE_FILE}'...")
    G = ox.load_graphml(CACHE_FILE)
else:
    print(f"Baixando rede viária de '{CIDADE}'...")
    print("(Primeira execução: ~2-5 minutos dependendo da conexão)")
    G = ox.graph_from_place(
        CIDADE,
        network_type=NETWORK_TYPE,
        simplify=True,
        retain_all=False,
    )
    ox.save_graphml(G, CACHE_FILE)
    print(f"Cache salvo em '{CACHE_FILE}'")

# ─── Verificar atributos geográficos ─────────────────────────────────────────
nos_sem_geo = [n for n, d in G.nodes(data=True) if "x" not in d or "y" not in d]
if nos_sem_geo:
    G.remove_nodes_from(nos_sem_geo)
    print(f"⚠ {len(nos_sem_geo)} nós sem coordenadas removidos.")

# ─── Resumo ───────────────────────────────────────────────────────────────────
comp_fracos = nx.number_weakly_connected_components(G)
maior_comp  = max(nx.weakly_connected_components(G), key=len)

print(f"""
✓ Rede carregada com sucesso!

  Nós (interseções):    {G.number_of_nodes():>8,}
  Arestas (vias):       {G.number_of_edges():>8,}
  Componentes fracos:   {comp_fracos:>8,}
  Maior componente:     {len(maior_comp):>8,} nós
  Grafo dirigido:       {'Sim':>8}
""")

Baixando rede viária de 'Natal, Rio Grande do Norte, Brazil'...
(Primeira execução: ~2-5 minutos dependendo da conexão)
Cache salvo em 'natal_drive_bruto.graphml'

✓ Rede carregada com sucesso!

  Nós (interseções):      18,586
  Arestas (vias):         48,211
  Componentes fracos:          1
  Maior componente:       18,586 nós
  Grafo dirigido:            Sim



## Seção 2 — Métricas Estruturais (NetworkX)

### Decisão: grafo dirigido vs. não-dirigido

O OSMnx retorna um **MultiDiGraph** (grafo dirigido com múltiplas arestas paralelas possíveis). Para calcular as métricas, precisamos lidar com isso:

| Métrica | Grafo usado | Justificativa |
|---------|-------------|---------------|
| Grau (in/out) | MultiDiGraph original | Preserva informação direcional das vias |
| Grau total | Graph não-dirigido | Compatível com definições clássicas |
| Betweenness | Graph não-dirigido | Definida para grafos sem direção; mão dupla é maioria em Natal |
| Closeness | Graph não-dirigido | Idem |
| K-core | Graph não-dirigido | Algoritmo de peeling só existe para grafos simples |

**Conversão**: `MultiDiGraph → Graph` colapsa arestas paralelas (mão dupla → uma aresta).

### Sobre a amostragem do Betweenness (`k=500`)
O cálculo exato de betweenness para ~10.000 nós teria complexidade O(VE), levando dezenas de minutos no Colab. Com `k=500` pares de nós amostrados aleatoriamente, obtemos uma aproximação estatisticamente robusta em ~2 minutos, com erro relativo < 5% nos top nós.

In [26]:
# ─── Converter para grafo não-dirigido simples ────────────────────────────────
print("Convertendo MultiDiGraph → Graph não-dirigido...")
G_ud = nx.Graph(G.to_undirected())

# Remover self-loops (causam NetworkXNotImplemented no core_number)
selfloops = list(nx.selfloop_edges(G_ud))
if selfloops:
    print(f"  ⚠ Removendo {len(selfloops)} self-loop(s)...")
    G_ud.remove_edges_from(selfloops)

print(f"  {G_ud.number_of_nodes():,} nós | {G_ud.number_of_edges():,} arestas\n")

# ─── 1. GRAU ─────────────────────────────────────────────────────────────────
print("[1/5] Calculando grau...")
grau     = dict(G_ud.degree())
grau_in  = dict(G.in_degree())
grau_out = dict(G.out_degree())
vals_grau = list(grau.values())

# ─── 2. DISTRIBUIÇÃO DE GRAU ─────────────────────────────────────────────────
dist_grau   = Counter(vals_grau)
grau_medio  = np.mean(vals_grau)
grau_max    = max(vals_grau)
grau_min    = min(vals_grau)
grau_std    = np.std(vals_grau)

# ─── 3. HUBS ─────────────────────────────────────────────────────────────────
N_HUBS    = 20
hubs_grau = sorted(grau.items(), key=lambda x: x[1], reverse=True)[:N_HUBS]

# ─── 4. BETWEENNESS CENTRALITY ───────────────────────────────────────────────
BW_K = 500
print(f"[2/5] Calculando betweenness (k={BW_K}, pode levar ~2 min)...")
betweenness = nx.betweenness_centrality(G_ud, k=BW_K, normalized=True, seed=42)
hubs_bw = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:N_HUBS]

# ─── 5. CLOSENESS CENTRALITY ─────────────────────────────────────────────────
print("[3/5] Calculando closeness...")
closeness = nx.closeness_centrality(G_ud)

# ─── 6. CORE NUMBER ──────────────────────────────────────────────────────────
print("[4/5] Calculando k-core decomposition...")
core_number = nx.core_number(G_ud)
k_max       = max(core_number.values())
dist_core   = Counter(core_number.values())

# ─── 7. Adicionar atributos ao grafo ─────────────────────────────────────────
print("[5/5] Adicionando atributos aos nós...")
for n in G.nodes():
    G.nodes[n]["grau"]        = int(grau.get(n, 0))
    G.nodes[n]["grau_in"]     = int(grau_in.get(n, 0))
    G.nodes[n]["grau_out"]    = int(grau_out.get(n, 0))
    G.nodes[n]["betweenness"] = float(round(betweenness.get(n, 0.0), 8))
    G.nodes[n]["closeness"]   = float(round(closeness.get(n, 0.0), 6))
    G.nodes[n]["core_number"] = int(core_number.get(n, 0))

# ─── Resumo ───────────────────────────────────────────────────────────────────
import math
K_ESCOLHIDO = max(3, math.ceil(k_max * 0.6))
cv_grau = grau_std / grau_medio

print(f"""
✓ Métricas calculadas!

  Grau médio:          {grau_medio:.2f}  (std={grau_std:.2f}, CV={cv_grau:.3f})
  Grau máximo:         {grau_max}
  Grau mínimo:         {grau_min}
  K-core máximo:       {k_max}
  K escolhido (60%):   {K_ESCOLHIDO}

  Top-3 hubs por grau:
    {hubs_grau[0]}  {hubs_grau[1]}  {hubs_grau[2]}

  Top-3 por betweenness:
    {hubs_bw[0]}  {hubs_bw[1]}  {hubs_bw[2]}
""")

Convertendo MultiDiGraph → Graph não-dirigido...
  ⚠ Removendo 8 self-loop(s)...
  18,586 nós | 27,963 arestas

[1/5] Calculando grau...
[2/5] Calculando betweenness (k=500, pode levar ~2 min)...
[3/5] Calculando closeness...
[4/5] Calculando k-core decomposition...
[5/5] Adicionando atributos aos nós...

✓ Métricas calculadas!

  Grau médio:          3.01  (std=0.73, CV=0.242)
  Grau máximo:         6
  Grau mínimo:         1
  K-core máximo:       2
  K escolhido (60%):   3

  Top-3 hubs por grau:
    (582071917, 6)  (300389738, 5)  (301415379, 5)

  Top-3 por betweenness:
    (502853903, 0.3261351735719383)  (6415220871, 0.3241840626935251)  (11025804797, 0.3240446292067693)



## Seção 3 — Análises Iniciais e Respostas Iniciais

Esta seção responde explicitamente algumas questões com base nos dados calculados, que auxiliarão em todo o entendimento dos dados. As respostas são salvas também no arquivo `relatorio_analise.txt`.

In [27]:
# ─── Dados de sobreposição ────────────────────────────────────────────────────
top10_grau_ids  = {n for n, _ in hubs_grau[:10]}
top10_bw_ids    = {n for n, _ in hubs_bw[:10]}
overlap_grau_bw = top10_grau_ids & top10_bw_ids

kcore_sub          = nx.k_core(G_ud, k=K_ESCOLHIDO)
kcore_ids          = set(kcore_sub.nodes())
kcore_pct          = 100 * len(kcore_ids) / G_ud.number_of_nodes()
overlap_hubs_kcore = top10_grau_ids & kcore_ids
overlap_bw_kcore   = top10_bw_ids   & kcore_ids

# ─── Funções de exibição ──────────────────────────────────────────────────────
def secao(titulo):
    print(f"\n{'='*65}\n  {titulo}\n{'='*65}")

def texto(t):
    print(t)

# ─── Estatísticas gerais ──────────────────────────────────────────────────────
secao("ESTATÍSTICAS GERAIS")
texto(f"  Nós:             {G.number_of_nodes():,}")
texto(f"  Arestas:         {G.number_of_edges():,}")
texto(f"  Grau médio:      {grau_medio:.2f}  (CV={cv_grau:.3f})")
texto(f"  K-core máximo:   {k_max}")
texto(f"  K escolhido:     {K_ESCOLHIDO}")
texto(f"  Nós no k-core:   {len(kcore_ids):,} ({kcore_pct:.1f}% da rede)")


  ESTATÍSTICAS GERAIS
  Nós:             18,586
  Arestas:         48,211
  Grau médio:      3.01  (CV=0.242)
  K-core máximo:   2
  K escolhido:     3
  Nós no k-core:   0 (0.0% da rede)


## Seção 4 — Visualizações

Geramos quatro visualizações complementares para os nossos dados:

1. **Distribuição de grau** (linear + log-log): mostra se a rede tem cauda longa;
2. **Grau × Betweenness** (scatter com cor = core_number): responde à Pergunta 1;
3. **Distribuição de core number**: mostra a estrutura hierárquica da rede;
4. **Mapa folium interativo**: sobreposição geográfica de todas as métricas.

Rode as próximas células, para gerar os gráficos, e salvar na pasta "graficos" (se não estiver criada, será criada a pasta).

In [28]:
# ─── FIG 1: Distribuição de grau ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Distribuição de Grau — Malha Viária de Natal/RN",
             fontsize=14, fontweight="bold", y=1.02)

# Histograma linear
ax1 = axes[0]
bins = range(grau_min, grau_max + 2)
ax1.hist(vals_grau, bins=bins, color="#3498db", edgecolor="white", linewidth=0.5)
ax1.axvline(grau_medio, color="#e74c3c", linestyle="--", linewidth=1.8,
            label=f"Média: {grau_medio:.2f}")
ax1.set_xlabel("Grau do nó", fontsize=11)
ax1.set_ylabel("Frequência", fontsize=11)
ax1.set_title("Escala linear")
ax1.legend(); ax1.grid(axis="y", alpha=0.3)

# Log-log
ax2 = axes[1]
g_uniq = sorted(dist_grau.keys())
f_vals = [dist_grau[g] for g in g_uniq]
ax2.scatter(g_uniq, f_vals, color="#e74c3c", s=25, alpha=0.8, zorder=5)
ax2.set_xscale("log"); ax2.set_yscale("log")
log_g = np.log10([g for g in g_uniq if g > 0])
log_f = np.log10([f for g, f in zip(g_uniq, f_vals) if g > 0])
if len(log_g) > 2:
    coef = np.polyfit(log_g, log_f, 1)
    g_fit = np.linspace(min(log_g), max(log_g), 100)
    ax2.plot(10**g_fit, 10**np.polyval(coef, g_fit),
             "--", color="#f39c12", lw=2, label=f"γ ≈ {abs(coef[0]):.2f}")
ax2.set_xlabel("Grau (log)"); ax2.set_ylabel("Frequência (log)")
ax2.set_title("Escala log-log")
ax2.legend(); ax2.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.savefig("graficos/fig1_distribuicao_grau.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Fig 1 salva")

✓ Fig 1 salva


In [29]:
# ─── FIG 2: Grau × Betweenness ───────────────────────────────────────────────
nos_lista = list(grau.keys())
g_vals    = [grau[n] for n in nos_lista]
b_vals    = [betweenness[n] for n in nos_lista]
c_vals    = [core_number[n] for n in nos_lista]

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(g_vals, b_vals, c=c_vals, cmap="plasma",
                s=10, alpha=0.5, linewidths=0,
                norm=mcolors.Normalize(vmin=0, vmax=k_max))

# Destacar top-10 betweenness
for i, (no, bw) in enumerate(hubs_bw[:10]):
    g  = grau[no]
    ax.scatter(g, bw, color="#e74c3c", s=90, zorder=10,
               edgecolors="white", linewidths=1)
    ax.annotate(f"#{i+1}", (g, bw), textcoords="offset points",
                xytext=(5, 3), fontsize=8, color="#e74c3c", fontweight="bold")

plt.colorbar(sc, ax=ax, label="Core number")
ax.set_xlabel("Grau do nó", fontsize=12)
ax.set_ylabel("Betweenness centrality", fontsize=12)
ax.set_title("Grau × Betweenness  |  cor = core number\n"
             "Vermelho: top-10 betweenness (candidatos a estações)",
             fontsize=12, fontweight="bold")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("graficos/fig2_grau_vs_betweenness.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Fig 2 salva")

✓ Fig 2 salva


In [30]:
# ─── FIG 3: Distribuição de Core Number ──────────────────────────────────────
ks_uniq = sorted(dist_core.keys())
ks_freq = [dist_core[k] for k in ks_uniq]
normas  = [k / max(ks_uniq) for k in ks_uniq]
cmap    = plt.get_cmap("viridis")

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(ks_uniq, ks_freq,
              color=[cmap(c) for c in normas],
              edgecolor="none")

# Destacar k escolhido
for i, k in enumerate(ks_uniq):
    if k >= K_ESCOLHIDO:
        bars[i].set_edgecolor("#e74c3c")
        bars[i].set_linewidth(2)

ax.axvline(K_ESCOLHIDO - 0.5, color="#e74c3c", linestyle="--", lw=2,
           label=f"K escolhido = {K_ESCOLHIDO}")
ax.set_xlabel("Core number", fontsize=12)
ax.set_ylabel("Número de nós", fontsize=12)
ax.set_title(f"Distribuição de Core Number  |  k_max = {k_max}\n"
             f"Barras com borda vermelha: subgrafo k-core (k ≥ {K_ESCOLHIDO})",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=mcolors.Normalize(0, k_max))
plt.colorbar(sm, ax=ax, label="Core number")
plt.tight_layout()
plt.savefig("graficos/fig3_distribuicao_core.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Fig 3 salva")

✓ Fig 3 salva


In [ ]:
# ─── MAPA INTERATIVO (folium) ─────────────────────────────────────────────────
import folium
from folium.plugins import HeatMap

lats   = [d["y"] for _, d in G.nodes(data=True) if "y" in d]
lons   = [d["x"] for _, d in G.nodes(data=True) if "x" in d]
centro = [np.mean(lats), np.mean(lons)]

mapa = folium.Map(location=centro, zoom_start=12, tiles="CartoDB positron")

# Camada 1: Arestas da rede (desativada por padrão — pesada)
grupo_rede = folium.FeatureGroup(name="Rede viária completa", show=False)
for u, v in list(G.edges())[:5000]:  # limite para desempenho
    if "y" in G.nodes[u] and "y" in G.nodes[v]:
        folium.PolyLine(
            [[G.nodes[u]["y"], G.nodes[u]["x"]],
             [G.nodes[v]["y"], G.nodes[v]["x"]]],
            color="#7f8c8d", weight=0.5, opacity=0.35
        ).add_to(grupo_rede)
grupo_rede.add_to(mapa)

# Camada 2: K-core
grupo_kcore = folium.FeatureGroup(name=f"K-core (k≥{K_ESCOLHIDO})", show=True)
for u, v in kcore_sub.edges():
    if u in G.nodes and v in G.nodes:
        if "y" in G.nodes[u] and "y" in G.nodes[v]:
            folium.PolyLine(
                [[G.nodes[u]["y"], G.nodes[u]["x"]],
                 [G.nodes[v]["y"], G.nodes[v]["x"]]],
                color="#8e44ad", weight=1.8, opacity=0.7
            ).add_to(grupo_kcore)
grupo_kcore.add_to(mapa)

# Camada 3: Heatmap betweenness
bw_max  = max(betweenness.values()) or 1
heat_pts = [
    [G.nodes[n]["y"], G.nodes[n]["x"], bw / bw_max]
    for n, bw in betweenness.items()
    if n in G.nodes and "y" in G.nodes[n] and bw / bw_max > 0.01
]
grupo_heat = folium.FeatureGroup(name="Heatmap: Betweenness", show=True)
HeatMap(heat_pts, radius=12, blur=15, max_zoom=14,
        gradient={"0.2": "blue", "0.5": "yellow", "1.0": "red"}
).add_to(grupo_heat)
grupo_heat.add_to(mapa)

# Camada 4: Top-20 hubs betweenness
grupo_hubs = folium.FeatureGroup(name="Top-20 hubs (betweenness)", show=True)
for rank, (no, bw) in enumerate(hubs_bw[:20], 1):
    if no not in G.nodes or "y" not in G.nodes[no]:
        continue
    lat, lon = G.nodes[no]["y"], G.nodes[no]["x"]
    cn = core_number.get(no, 0)
    g  = grau.get(no, 0)
    folium.CircleMarker(
        location=[lat, lon], radius=max(5, 20 - rank),
        color="white", weight=1.5, fill=True,
        fill_color="#e74c3c", fill_opacity=0.85,
        popup=folium.Popup(
            f"<b>#{rank} Betweenness Hub</b><br>"
            f"Nó OSM: {no}<br>BW: {bw:.6f}<br>"
            f"Grau: {g}<br>Core: {cn}<br>"
            f"({lat:.5f}, {lon:.5f})", max_width=230),
        tooltip=f"#{rank} bw={bw:.5f}"
    ).add_to(grupo_hubs)
grupo_hubs.add_to(mapa)

folium.LayerControl(collapsed=False).add_to(mapa)

legenda = """
<div style="position:fixed;bottom:30px;left:30px;z-index:9999;
background:white;padding:12px 16px;border-radius:8px;
box-shadow:0 2px 8px rgba(0,0,0,.2);font-size:12px;">
<b>Legenda</b><br>
<span style="color:#8e44ad">━━</span> K-core (núcleo)<br>
<span style="color:#e74c3c">●</span> Top-20 betweenness<br>
<i>Clique nos pontos para detalhes</i></div>
"""
mapa.get_root().html.add_child(folium.Element(legenda))
mapa.save("mapa_natal.html")

print("✓ Mapa salvo em 'mapa_natal.html'")
print("  Abra o arquivo no navegador ou use:")
print("  from IPython.display import IFrame")
print("  IFrame('mapa_natal.html', width=900, height=500)")

✓ Mapa salvo em 'mapa_natal.html'
  Abra o arquivo no navegador ou use:
  from IPython.display import IFrame
  IFrame('mapa_natal.html', width=900, height=500)


#### Aproveite o mapa gerado (mapa_natal.html), e abra-o no navegador para ver a estrutura inicial do mapa interativo com Folium.

## Seção 5 — Exportação para Gephi (.graphml)

O Gephi importa grafos no formato `.graphml`. Para que as métricas fiquem disponíveis no Gephi como atributos dos nós (para colorir e dimensionar os nós), é necessário:

1. Garantir que os atributos estão no grafo (já feito na Seção 2);
2. **Converter tipos numpy para Python nativo**, pois o `networkx.write_graphml` exige `int`, `float`, `str` puros;
3. Salvar com `nx.write_graphml()`.

Após importar no Gephi, vamos usar:
- **Appearance → Nodes → Size → Ranking → `grau`** para tamanho;
- **Appearance → Nodes → Color → Ranking → `core_number`** para cor;
- **Layout → Geo Layout** com `longitude=longitude`, `latitude=latitude` para vista geográfica;
- **Layout → ForceAtlas2** para vista estrutural.

In [ ]:
def converter_atributos(G):
    G2 = G.copy()
    def _conv(val):
        if isinstance(val, np.integer):  return int(val)
        if isinstance(val, np.floating): return float(val)
        if isinstance(val, np.bool_):    return bool(val)
        if not isinstance(val, (int, float, str, bool)): return str(val)
        return val
    for n, data in G2.nodes(data=True):
        for k, v in list(data.items()): data[k] = _conv(v)
    for u, v, k, data in G2.edges(keys=True, data=True):
        for attr, val in list(data.items()): data[attr] = _conv(val)
    return G2

print("Preparando grafo completo para exportação...")
G_export = converter_atributos(G)

# IDs únicos nas arestas (evita erro de merge no Gephi)
for i, (u, v, k) in enumerate(G_export.edges(keys=True)):
    G_export[u][v][k]["id"] = f"e{i}"

# ─── Garantir x/y como float explícito para o Geo Layout ─────────────────────
for n, data in G_export.nodes(data=True):
    if "x" in data:
        data["longitude"] = float(data["x"])   # copia com nome alternativo
        data["latitude"]  = float(data["y"])

ARQUIVO_GRAPHML = "natal_drive_completo.graphml"
nx.write_graphml(G_export, ARQUIVO_GRAPHML, edge_id_from_attribute="id")

tamanho = os.path.getsize(ARQUIVO_GRAPHML) / (1024*1024)
print(f"""
✓ Exportado: {ARQUIVO_GRAPHML} ({tamanho:.1f} MB)
  Nós:     {G_export.number_of_nodes():,}
  Arestas: {G_export.number_of_edges():,}

  Atributos nos nós (usados no Gephi):
    x            → longitude  (Geo Layout)
    y            → latitude   (Geo Layout)
    grau         → tamanho do nó
    core_number  → cor do nó
    betweenness  → destaque / filtro
""")

Preparando grafo completo para exportação...

✓ Exportado: natal_drive_completo.graphml (23.8 MB)
  Nós:     18,586
  Arestas: 48,211

  Atributos nos nós (usados no Gephi):
    x            → longitude  (Geo Layout)
    y            → latitude   (Geo Layout)
    grau         → tamanho do nó
    core_number  → cor do nó
    betweenness  → destaque / filtro



## Seção 6 — Geração do Mapa de Melhores Pontos para Estação de Metrô

Rodando a célula abaixo, vamos gerar agora outro mapa em formato html (mapa_metro_proposto.html); Este mapa mostrará os melhores N pontos (podendo escolher a quantidade de estações mudando o número da variável "N_ESTACOES") para se ter estações de um eventual metro, junto das ligações entre as estações. Conforme subimos ou descemos o número de estações, percebemos que algumas poucas regiões eventualmente possuem mais que uma estação "bem próximas", o que mostra que aquela região pode ser uma região cujo haja bastante necessidade de fluxo.

In [ ]:
import folium
import numpy as np
import networkx as nx
from sklearn.cluster import KMeans

# ─── 1. Selecionar candidatos por betweenness ─────────────────────────────────
# Top 2% por betweenness = nós estruturalmente mais críticos para fluxo
p98 = np.percentile(list(betweenness.values()), 98)
candidatos = {
    n: betweenness[n]
    for n in G.nodes()
    if betweenness[n] >= p98 and "y" in G.nodes[n]
}
print(f"Candidatos (top 2% betweenness): {len(candidatos)} nós")

# ─── 2. Clusterizar geograficamente em N grupos (= N estações) ────────────────
N_ESTACOES = 12   # ajuste conforme desejar

coords = np.array([[G.nodes[n]["y"], G.nodes[n]["x"]] for n in candidatos])
ids    = list(candidatos.keys())

kmeans = KMeans(n_clusters=N_ESTACOES, random_state=42, n_init=10)
kmeans.fit(coords)

# Para cada cluster, pegar o nó de maior betweenness como representante
clusters = {}
for idx, label in enumerate(kmeans.labels_):
    no = ids[idx]
    bw = candidatos[no]
    if label not in clusters or bw > clusters[label]["bw"]:
        clusters[label] = {
            "no":  no,
            "bw":  bw,
            "lat": G.nodes[no]["y"],
            "lon": G.nodes[no]["x"],
            "grau": grau.get(no, 0),
            "core": core_number.get(no, 0),
        }

estacoes = sorted(clusters.values(), key=lambda e: e["bw"], reverse=True)

print(f"\n{'─'*60}")
print(f"  {'#':<4} {'Betweenness':>12}  {'Grau':>5}  {'Core':>5}  {'Lat':>10}  {'Lon':>11}")
print(f"{'─'*60}")
for i, e in enumerate(estacoes, 1):
    print(f"  {i:<4} {e['bw']:>12.6f}  {e['grau']:>5}  {e['core']:>5}  "
          f"{e['lat']:>10.5f}  {e['lon']:>11.5f}")

# ─── 3. Mapa focado nas estações propostas ────────────────────────────────────
lats   = [e["lat"] for e in estacoes]
lons   = [e["lon"] for e in estacoes]
centro = [np.mean(lats), np.mean(lons)]

mapa_metro = folium.Map(location=centro, zoom_start=12, tiles="CartoDB positron")

# Arestas do k-core como pano de fundo
grupo_kcore = folium.FeatureGroup(name=f"K-core (k≥{K_ESCOLHIDO})", show=True)
for u, v in kcore_sub.edges():
    if u in G.nodes and v in G.nodes:
        if "y" in G.nodes[u] and "y" in G.nodes[v]:
            folium.PolyLine(
                [[G.nodes[u]["y"], G.nodes[u]["x"]],
                 [G.nodes[v]["y"], G.nodes[v]["x"]]],
                color="#bdc3c7", weight=1, opacity=0.5
            ).add_to(grupo_kcore)
grupo_kcore.add_to(mapa_metro)

# Linha conectando as estações em ordem de betweenness (corredor proposto)
coords_linha = [[e["lat"], e["lon"]] for e in estacoes]
folium.PolyLine(
    coords_linha,
    color="#e74c3c", weight=4, opacity=0.8,
    tooltip="Corredor proposto (ordem por betweenness)"
).add_to(mapa_metro)

# Marcadores das estações
cores_rank = ["#c0392b","#e74c3c","#e67e22","#f39c12","#f1c40f",
              "#2ecc71","#27ae60","#1abc9c","#3498db","#2980b9","#9b59b6","#8e44ad"]

for i, e in enumerate(estacoes, 1):
    cor = cores_rank[min(i-1, len(cores_rank)-1)]
    folium.CircleMarker(
        location=[e["lat"], e["lon"]],
        radius=14,
        color="white", weight=2,
        fill=True, fill_color=cor, fill_opacity=0.92,
        popup=folium.Popup(
            f"<b>Estação #{i}</b><br>"
            f"Betweenness: {e['bw']:.6f}<br>"
            f"Grau: {e['grau']}<br>"
            f"Core number: {e['core']}<br>"
            f"({e['lat']:.5f}, {e['lon']:.5f})",
            max_width=220
        ),
        tooltip=f"Estação #{i}  bw={e['bw']:.5f}"
    ).add_to(mapa_metro)

    # Número da estação visível no mapa
    folium.Marker(
        location=[e["lat"], e["lon"]],
        icon=folium.DivIcon(
            html=f'<div style="font-size:10px;font-weight:bold;color:white;'
                 f'text-align:center;margin-top:4px;">{i}</div>',
            icon_size=(20, 20), icon_anchor=(10, 10)
        )
    ).add_to(mapa_metro)

folium.LayerControl().add_to(mapa_metro)
mapa_metro.save("mapa_metro_proposto.html")
print("\n✓ Mapa salvo: mapa_metro_proposto.html")
print("  Abra no navegador — clique em cada estação para ver as métricas.")

Candidatos (top 2% betweenness): 372 nós

────────────────────────────────────────────────────────────
  #     Betweenness   Grau   Core         Lat          Lon
────────────────────────────────────────────────────────────
  1        0.326135      3      2    -5.79728    -35.23863
  2        0.324184      3      2    -5.77770    -35.24937
  3        0.229549      3      2    -5.76949    -35.27040
  4        0.152328      4      2    -5.82164    -35.21357
  5        0.145149      4      2    -5.80453    -35.22753
  6        0.127682      3      2    -5.77459    -35.19459
  7        0.125072      4      2    -5.81652    -35.21171
  8        0.121547      3      2    -5.81984    -35.25137
  9        0.109562      3      2    -5.77825    -35.19383
  10       0.092316      3      2    -5.83703    -35.24574
  11       0.089273      3      2    -5.74834    -35.21016
  12       0.085184      3      2    -5.74416    -35.28979
  13       0.081065      3      2    -5.86124    -35.19169
  14      

## ✅ Resumo Final

| Arquivo gerado | Descrição |
|---|---|
| `natal_drive.graphml` | Importe no Gephi para visualizações |
| `mapa_natal.html` | Mapa interativo — abra no navegador |
| `graficos/fig1_*.png` | Distribuição de grau (linear + log-log) |
| `graficos/fig2_*.png` | Scatter grau × betweenness |
| `graficos/fig3_*.png` | Distribuição de core number |
| `relatorio_analise.txt` | Respostas completas às 7 perguntas |

---

### 🚇 Conclusão sobre o traçado de metrô

A análise estrutural sugere que um corredor de metrô em Natal deve priorizar:

1. **Nós de alto betweenness** — pontos de máxima intermediação na rede, que concentram fluxo de toda a cidade;
2. **Travessia do Potengi** — o gargalo estrutural mais crítico, onde qualquer intervenção tem impacto desproporcional na mobilidade;
3. **Cobertura do k-core periférico** — bairros fora do núcleo denso têm menos alternativas de rota e maior dependência de transporte público.

Esses três critérios, derivados diretamente das métricas de grafo, apontam para um traçado **Norte-Sul** como o mais impactante estruturalmente — consistente com os projetos de mobilidade urbana historicamente estudados para a cidade.